# Whisper Inference Optimization for Real-Time Banking Voice Commands

Final analysis notebook for:
- Baseline ASR comparison
- Whisper optimization comparison
- Faster-Whisper Small vs Medium
- Latency–accuracy trade-off
- Confidence intervals, outliers and paired significance testing
- Best practical configuration selection


In [ ]:
from pathlib import Path
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)


## 1. Load and standardise results


In [ ]:
def clean_name(x):
    return re.sub(r"[^a-z0-9]+", "_", str(x).strip().lower()).strip("_")

def first_col(columns, choices):
    return next((x for x in choices if x in columns), None)

def pretty_model(x):
    key = clean_name(x)
    names = {
        "whisperadapter":"Whisper Baseline",
        "whisper":"Whisper Baseline",
        "whisper_baseline":"Whisper Baseline",
        "facebook_wav2vec2_base_960h":"Wav2Vec2",
        "wav2vec2":"Wav2Vec2",
        "voskadapter":"Vosk",
        "vosk":"Vosk",
        "whisper_greedy":"Whisper Greedy",
        "whisper_no_context":"Whisper No Context",
        "whisper_vad":"Whisper + VAD",
        "whisper_fp16":"Whisper FP16",
        "whisper_int8":"Faster-Whisper INT8",
        "faster_whisper":"Faster-Whisper Small",
        "faster_whisper_small":"Faster-Whisper Small",
        "faster_whisper_medium":"Faster-Whisper Medium",
        "whisper_limited_tokens":"Whisper Limited Tokens",
        "whisper_no_timestamps":"Whisper No Timestamps",
        "whisper_no_fallback":"Whisper No Fallback",
        "whisper_direct_decode":"Whisper Direct Decode",
        "whisper_best":"Best Combined Configuration",
        "best_combined":"Best Combined Configuration",
    }
    return names.get(key, str(x).replace("_", " ").title())

candidates = [
    "combined_per_file.csv",
    "all_results.csv",
    "merged_results.csv",
    "benchmark_results.csv",
    "per_file_results.csv",
]

path = next((RESULTS_DIR / f for f in candidates if (RESULTS_DIR / f).exists()), None)

if path is None:
    raise FileNotFoundError(
        "Place the merged per-file CSV in results/. Expected one of: "
        + ", ".join(candidates)
    )

per_file = pd.read_csv(path)
per_file.columns = [clean_name(x) for x in per_file.columns]

model_col = first_col(per_file.columns, ["model","model_name","adapter","experiment","configuration"])
latency_col = first_col(per_file.columns, ["latency_ms","inference_latency_ms","elapsed_ms","duration_ms","total_latency_ms"])
wer_col = first_col(per_file.columns, ["wer","word_error_rate"])
file_col = first_col(per_file.columns, ["file","filename","audio_file","audio_path","sample_id"])
reference_col = first_col(per_file.columns, ["reference","canonical_transcript","reference_text","ground_truth","target"])
prediction_col = first_col(per_file.columns, ["transcript","prediction","predicted_text","hypothesis","text"])

required = {"model":model_col, "latency":latency_col, "wer":wer_col, "file":file_col}
missing = [k for k,v in required.items() if v is None]
if missing:
    raise KeyError(f"Missing columns: {missing}. Available: {list(per_file.columns)}")

rename = {model_col:"model", latency_col:"latency_ms", wer_col:"wer", file_col:"file"}
if reference_col: rename[reference_col] = "reference"
if prediction_col: rename[prediction_col] = "prediction"

per_file = per_file.rename(columns=rename)
per_file["latency_ms"] = pd.to_numeric(per_file["latency_ms"], errors="coerce")
per_file["wer"] = pd.to_numeric(per_file["wer"], errors="coerce")
per_file["model_pretty"] = per_file["model"].astype(str).map(pretty_model)
per_file = per_file.dropna(subset=["latency_ms","wer"]).copy()

print("Loaded:", path)
print("Rows:", len(per_file))
display(per_file.head())
display(per_file[["model","model_pretty"]].drop_duplicates().sort_values("model_pretty"))


## 2. Dataset overview


In [ ]:
overview = pd.DataFrame({
    "Metric":["Unique audio files","Model configurations","Benchmark observations"],
    "Value":[per_file["file"].nunique(), per_file["model_pretty"].nunique(), len(per_file)]
})
display(overview)

display(
    per_file.groupby("model_pretty", as_index=False)
    .agg(observations=("file","size"), unique_files=("file","nunique"))
    .sort_values("model_pretty")
)


## 3. Final summary table


In [ ]:
def bootstrap_ci(values, statistic=np.median, iterations=5000, seed=42):
    values = pd.Series(values).dropna().to_numpy(float)
    if len(values) == 0:
        return np.nan, np.nan
    rng = np.random.default_rng(seed)
    stats = [statistic(rng.choice(values, size=len(values), replace=True)) for _ in range(iterations)]
    return tuple(np.percentile(stats, [2.5, 97.5]))

rows = []
for model, group in per_file.groupby("model_pretty"):
    lo, hi = bootstrap_ci(group["latency_ms"])
    rows.append({
        "model":model,
        "n_files":group["file"].nunique(),
        "mean_latency_ms":group["latency_ms"].mean(),
        "median_latency_ms":group["latency_ms"].median(),
        "p95_latency_ms":group["latency_ms"].quantile(.95),
        "std_latency_ms":group["latency_ms"].std(ddof=1),
        "median_latency_ci_low_ms":lo,
        "median_latency_ci_high_ms":hi,
        "mean_wer":group["wer"].mean(),
        "median_wer":group["wer"].median(),
        "corpus_wer":group["wer"].mean(),
    })

summary = pd.DataFrame(rows).sort_values(["corpus_wer","median_latency_ms"]).reset_index(drop=True)
display(summary.round(4))


## 4. Baseline ASR comparison


In [ ]:
baseline_models = ["Whisper Baseline","Wav2Vec2","Vosk"]
baseline_summary = summary[summary["model"].isin(baseline_models)].sort_values("median_latency_ms")
display(baseline_summary.round(4))

if not baseline_summary.empty:
    plt.figure(figsize=(8,5))
    plt.bar(baseline_summary["model"], baseline_summary["median_latency_ms"])
    plt.title("Median Latency: Baseline ASR Models")
    plt.xlabel("Model")
    plt.ylabel("Median latency (ms)")
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(8,5))
    plt.bar(baseline_summary["model"], baseline_summary["corpus_wer"])
    plt.title("Corpus WER: Baseline ASR Models")
    plt.xlabel("Model")
    plt.ylabel("WER")
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.show()


## 5. Whisper optimization comparison


In [ ]:
whisper_summary = summary[summary["model"].str.contains("Whisper", case=False, na=False)].sort_values("median_latency_ms")
display(whisper_summary.round(4))

if not whisper_summary.empty:
    plt.figure(figsize=(11,6))
    plt.bar(whisper_summary["model"], whisper_summary["median_latency_ms"])
    plt.title("Median Latency Across Whisper Configurations")
    plt.xlabel("Configuration")
    plt.ylabel("Median latency (ms)")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(11,6))
    plt.bar(whisper_summary["model"], whisper_summary["corpus_wer"])
    plt.title("WER Across Whisper Configurations")
    plt.xlabel("Configuration")
    plt.ylabel("WER")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()


## 6. Latency–accuracy trade-off


In [ ]:
plt.figure(figsize=(10,6))
plt.scatter(summary["median_latency_ms"], summary["corpus_wer"])

for _, row in summary.iterrows():
    plt.annotate(
        row["model"],
        (row["median_latency_ms"], row["corpus_wer"]),
        xytext=(6,5),
        textcoords="offset points",
        fontsize=8
    )

plt.title("Latency–Accuracy Trade-off")
plt.xlabel("Median latency (ms)")
plt.ylabel("WER")
plt.tight_layout()
plt.show()


## 7. Latency distributions and outliers


In [ ]:
order = summary.sort_values("median_latency_ms")["model"].tolist()
groups = [
    per_file.loc[per_file["model_pretty"] == model, "latency_ms"].dropna().to_numpy()
    for model in order
]

plt.figure(figsize=(12,6))
plt.boxplot(groups, tick_labels=order, showfliers=True)
plt.title("Latency Distribution by Configuration")
plt.xlabel("Configuration")
plt.ylabel("Latency (ms)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

outliers = []
for model, group in per_file.groupby("model_pretty"):
    q1, q3 = group["latency_ms"].quantile([.25,.75])
    limit = q3 + 1.5 * (q3-q1)
    temp = group[group["latency_ms"] > limit].copy()
    temp["outlier_threshold_ms"] = limit
    outliers.append(temp)

latency_outliers = pd.concat(outliers, ignore_index=True) if outliers else pd.DataFrame()
if latency_outliers.empty:
    print("No IQR-based latency outliers detected.")
else:
    display(
        latency_outliers[
            ["model_pretty","file","latency_ms","outlier_threshold_ms","wer"]
        ].sort_values("latency_ms", ascending=False)
    )


## 8. Faster-Whisper Small vs Medium


In [ ]:
sizes = summary[summary["model"].isin(["Faster-Whisper Small","Faster-Whisper Medium"])].sort_values("median_latency_ms")
display(sizes.round(4))

if len(sizes) == 2:
    plt.figure(figsize=(8,5))
    plt.bar(sizes["model"], sizes["median_latency_ms"])
    plt.title("Faster-Whisper Small vs Medium: Median Latency")
    plt.ylabel("Median latency (ms)")
    plt.xticks(rotation=15, ha="right")
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(8,5))
    plt.bar(sizes["model"], sizes["corpus_wer"])
    plt.title("Faster-Whisper Small vs Medium: WER")
    plt.ylabel("WER")
    plt.xticks(rotation=15, ha="right")
    plt.tight_layout()
    plt.show()

    small = sizes[sizes["model"] == "Faster-Whisper Small"].iloc[0]
    medium = sizes[sizes["model"] == "Faster-Whisper Medium"].iloc[0]

    comparison = pd.DataFrame({
        "Metric":["Median latency change (%)","WER change (%)"],
        "Medium relative to Small":[
            (medium["median_latency_ms"]-small["median_latency_ms"]) / small["median_latency_ms"] * 100,
            (medium["corpus_wer"]-small["corpus_wer"]) / small["corpus_wer"] * 100 if small["corpus_wer"] else np.nan
        ]
    })
    display(comparison.round(2))
else:
    print("Run and merge both Faster-Whisper Small and Medium results.")


## 9. Comparison with Whisper baseline


In [ ]:
baseline_name = "Whisper Baseline"

if baseline_name in summary["model"].values:
    baseline = summary[summary["model"] == baseline_name].iloc[0]
    comparison = summary.copy()
    comparison["median_latency_reduction_pct"] = (
        baseline["median_latency_ms"] - comparison["median_latency_ms"]
    ) / baseline["median_latency_ms"] * 100
    comparison["wer_change_pct"] = (
        comparison["corpus_wer"] - baseline["corpus_wer"]
    ) / baseline["corpus_wer"] * 100 if baseline["corpus_wer"] else np.nan

    display(
        comparison[
            ["model","median_latency_ms","corpus_wer",
             "median_latency_reduction_pct","wer_change_pct"]
        ].sort_values("median_latency_reduction_pct", ascending=False).round(2)
    )
else:
    print("Whisper Baseline was not found.")


## 10. Paired latency significance test


In [ ]:
try:
    from scipy.stats import wilcoxon

    tests = []
    base = (
        per_file[per_file["model_pretty"] == baseline_name][["file","latency_ms"]]
        .rename(columns={"latency_ms":"baseline_latency_ms"})
    )

    for model in sorted(per_file["model_pretty"].unique()):
        if model == baseline_name:
            continue

        candidate = (
            per_file[per_file["model_pretty"] == model][["file","latency_ms"]]
            .rename(columns={"latency_ms":"candidate_latency_ms"})
        )

        paired = base.merge(candidate, on="file", how="inner")
        if len(paired) < 5:
            continue

        diff = paired["candidate_latency_ms"] - paired["baseline_latency_ms"]
        if np.allclose(diff, 0):
            stat, p = 0.0, 1.0
        else:
            stat, p = wilcoxon(
                paired["candidate_latency_ms"],
                paired["baseline_latency_ms"]
            )

        tests.append({
            "configuration":model,
            "paired_samples":len(paired),
            "median_difference_ms":diff.median(),
            "wilcoxon_statistic":stat,
            "p_value":p,
            "significant_at_0_05":p < .05
        })

    significance = pd.DataFrame(tests)
    display(significance.sort_values("p_value").round(6) if not significance.empty else significance)

except ImportError:
    significance = pd.DataFrame()
    print("Install SciPy first: pip install scipy")


## 11. Inspect worst transcription errors


In [ ]:
cols = ["model_pretty","file","wer","latency_ms"]
if "reference" in per_file.columns: cols.append("reference")
if "prediction" in per_file.columns: cols.append("prediction")

display(
    per_file[cols]
    .sort_values(["wer","latency_ms"], ascending=[False,False])
    .head(25)
)


## 12. Optional number-related banking command analysis


In [ ]:
NUMBER_WORDS = {
    "zero","one","two","three","four","five","six","seven","eight","nine",
    "ten","eleven","twelve","thirteen","fourteen","fifteen","sixteen",
    "seventeen","eighteen","nineteen","twenty","thirty","forty","fifty",
    "sixty","seventy","eighty","ninety","hundred","thousand","million"
}

def has_number(text):
    tokens = re.findall(r"[a-z0-9]+", str(text).lower())
    return any(t.isdigit() or t in NUMBER_WORDS for t in tokens)

if "reference" in per_file.columns:
    numeric = per_file[per_file["reference"].map(has_number)].copy()
    numeric_summary = (
        numeric.groupby("model_pretty", as_index=False)
        .agg(
            numeric_command_count=("file","size"),
            mean_numeric_command_wer=("wer","mean"),
            median_numeric_command_wer=("wer","median")
        )
        .sort_values("mean_numeric_command_wer")
    )
    display(numeric_summary.round(4))
else:
    numeric_summary = pd.DataFrame()
    print("Reference transcripts are required.")


## 13. Select the best practical configuration


In [ ]:
MAX_RELATIVE_WER_INCREASE = 0.10

if baseline_name in summary["model"].values:
    baseline_wer = summary.loc[summary["model"] == baseline_name, "corpus_wer"].iloc[0]
    threshold = baseline_wer * (1 + MAX_RELATIVE_WER_INCREASE)
    eligible = summary[summary["corpus_wer"] <= threshold].copy()

    if eligible.empty:
        best = None
        print("No configuration met the WER threshold.")
    else:
        best = eligible.sort_values(["median_latency_ms","corpus_wer"]).iloc[0]
        print("Best practical configuration:", best["model"])
        print("Median latency:", round(best["median_latency_ms"], 2), "ms")
        print("WER:", round(best["corpus_wer"], 4))
        print("Maximum accepted WER:", round(threshold, 4))
else:
    best = None
    print("Whisper Baseline is required.")


## 14. Export final tables


In [ ]:
summary.to_csv(RESULTS_DIR / "final_experiment_summary.csv", index=False)
baseline_summary.to_csv(RESULTS_DIR / "baseline_model_comparison.csv", index=False)
whisper_summary.to_csv(RESULTS_DIR / "whisper_optimization_comparison.csv", index=False)
sizes.to_csv(RESULTS_DIR / "faster_whisper_small_vs_medium.csv", index=False)

if "significance" in globals() and not significance.empty:
    significance.to_csv(RESULTS_DIR / "latency_significance_tests.csv", index=False)

if "numeric_summary" in globals() and not numeric_summary.empty:
    numeric_summary.to_csv(RESULTS_DIR / "numeric_command_error_summary.csv", index=False)

print("Tables exported to:", RESULTS_DIR.resolve())


## Dissertation interpretation template

> The experiments evaluated multiple ASR systems and Whisper inference configurations using the same short banking voice-command dataset and hardware environment. Performance was measured using mean, median and 95th-percentile inference latency together with Word Error Rate. The best practical configuration achieved a median latency of **[value] ms** and a WER of **[value]**, compared with **[value] ms** and **[value]** for the Whisper baseline. This corresponds to a latency change of **[value]%** while maintaining an acceptable accuracy level. The results indicate that **[configuration]** offers the most suitable latency–accuracy trade-off for the evaluated banking voice-command scenario.

Do not conclude that a configuration is better from the mean alone. Review the median, P95, confidence interval, paired test, outliers and transcription examples.
